## mini-project - (Project one - predicting the fuel efficiency of a car)

#### the data Link : 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'


### WE take only sepcific features :
    1 . no of cylinders
    2 .  displacement horsepower
    3 .  weight
    4 . acceleration
    5 . MPG
    6 . Model_year
    7 . Origin


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sklearn 
import sklearn.model_selection

## Loading the data set and explore , clean for model training

In [2]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

df = pd.read_csv(url, names=column_names,
                 na_values = "?", comment='\t',
                 sep=" ", skipinitialspace=True)

df.tail()

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
393,27.0,4,140.0,86.0,2790.0,15.6,82,1
394,44.0,4,97.0,52.0,2130.0,24.6,82,2
395,32.0,4,135.0,84.0,2295.0,11.6,82,1
396,28.0,4,120.0,79.0,2625.0,18.6,82,1
397,31.0,4,119.0,82.0,2720.0,19.4,82,1


In [3]:
# drop the null values

df = df.dropna()
df = df.reset_index(drop=True)



In [4]:
# Train / test spilt 

df_train,df_test = sklearn.model_selection.train_test_split(
    df, train_size=0.8, random_state=1
    
)


In [5]:
# Describe the train stats

train_stats = df_train.describe().transpose()
print(train_stats)

              count         mean         std     min     25%     50%     75%  \
MPG           313.0    23.404153    7.666909     9.0    17.5    23.0    29.0   
Cylinders     313.0     5.402556    1.701506     3.0     4.0     4.0     8.0   
Displacement  313.0   189.512780  102.675646    68.0   104.0   140.0   260.0   
Horsepower    313.0   102.929712   37.919046    46.0    75.0    92.0   120.0   
Weight        313.0  2961.198083  848.602146  1613.0  2219.0  2755.0  3574.0   
Acceleration  313.0    15.704473    2.725399     8.5    14.0    15.5    17.3   
Model Year    313.0    75.929712    3.675305    70.0    73.0    76.0    79.0   
Origin        313.0     1.591054    0.807923     1.0     1.0     1.0     2.0   

                 max  
MPG             46.6  
Cylinders        8.0  
Displacement   455.0  
Horsepower     230.0  
Weight        5140.0  
Acceleration    24.8  
Model Year      82.0  
Origin           3.0  


In [6]:
# Picking only numeric columns:

numeric_column_names = ['Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration']

In [7]:
# Now normalize the data :



df_train_norm = df_train.copy()
df_test_norm  = df_test.copy()

for col_name in numeric_column_names:
    mean = train_stats.loc[col_name, 'mean']
    std  = train_stats.loc[col_name, 'std']

    if std != 0:
        df_train_norm[col_name] = (df_train_norm[col_name] - mean) / std
        df_test_norm[col_name]  = (df_test_norm[col_name] - mean) / std



In [8]:
df_train_norm

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
334,27.2,-0.824303,-0.530922,-0.499214,-0.555264,-0.001641,81,1
258,18.6,0.351127,0.345625,0.186457,0.776338,1.099115,78,1
139,29.0,-0.824303,-0.891280,-0.525586,-0.874613,0.291894,74,2
310,37.2,-0.824303,-1.008153,-1.000281,-1.110294,0.255202,80,3
349,33.0,-0.824303,-0.823104,-0.762934,-0.908786,-0.552019,81,2
...,...,...,...,...,...,...,...,...
203,28.0,-0.824303,-0.901020,-0.736562,-0.950031,0.255202,76,3
255,19.4,0.351127,0.413800,-0.340982,0.293190,0.548737,78,1
72,13.0,1.526556,1.144256,0.713897,1.339617,-0.625403,72,1
235,30.5,-0.824303,-0.891280,-1.053025,-1.072585,0.475353,77,1


## Create a Bucket for model years:

### Bucket   = {
          0 if year < 73,
          1 if 73 <= year < 76
          2 if 76 <= year < 79
          3 if year >= 79
             
}

In [9]:
boundaries = torch.tensor([73,76,79])
v = torch.tensor(df_train_norm['Model Year'].values)
df_train_norm['Model Year Bucketed'] = torch.bucketize(
    v, boundaries, right=True
)



v = torch.tensor(df_test_norm['Model Year'].values)
df_test_norm['Model Year Bucketed'] = torch.bucketize(
    v, boundaries, right=True
)

numeric_column_names.append('Model Year Bucketed')

In [10]:
numeric_column_names

['Cylinders',
 'Displacement',
 'Horsepower',
 'Weight',
 'Acceleration',
 'Model Year Bucketed']

## WE use torch.nn.functional one hot for this project

In [11]:
from torch.nn.functional import one_hot

total_orgin = len(set(df_train_norm['Origin']))
origin_encoded = one_hot(torch.from_numpy(
    df_train_norm['Origin'].values) % total_orgin)

x_train_numeric = torch.tensor(
    df_train_norm[numeric_column_names].values)

x_train = torch.cat([x_train_numeric,origin_encoded],1).float()

origin_encoded = one_hot(torch.from_numpy(
    df_test_norm['Origin'].values) % total_orgin)

x_test_numeric = torch.tensor(
    df_test_norm[numeric_column_names].values
)

x_test = torch.cat([x_test_numeric,origin_encoded],1).float()



###   Creating Label Tensors

In [12]:
y_train = torch.tensor(df_train_norm['MPG'].values).float()
y_test = torch.tensor(df_test_norm['MPG'].values).float()

In [13]:
print(y_train)

tensor([27.2000, 18.6000, 29.0000, 37.2000, 33.0000, 13.0000, 22.4000, 44.6000,
        27.4000, 31.0000, 24.0000, 28.0000, 17.6000, 15.0000, 11.0000, 12.0000,
        26.8000, 13.0000, 21.6000, 20.8000, 16.9000, 20.0000, 26.5000, 18.0000,
        26.0000, 24.0000, 29.5000, 20.2000, 32.8000, 14.0000, 16.0000, 31.8000,
        30.7000, 34.1000, 20.0000, 15.0000, 28.4000, 20.0000, 13.0000, 35.0000,
        31.0000, 44.0000, 27.0000, 17.0000, 34.1000, 14.0000, 23.0000, 28.1000,
        14.5000, 24.0000, 29.9000, 36.0000, 23.9000, 15.0000, 16.0000, 20.0000,
        16.0000, 24.0000, 23.9000, 11.0000, 14.0000, 19.0000, 30.0000, 21.0000,
        15.0000, 23.0000, 18.0000, 29.0000, 18.0000, 12.0000, 33.0000, 29.5000,
        19.0000, 14.0000, 19.0000, 26.6000, 36.0000, 38.0000, 24.0000, 13.0000,
        16.0000, 17.5000, 19.0000, 16.0000, 27.0000, 16.5000, 20.3000, 24.0000,
        30.0000, 23.0000, 17.0000, 18.0000, 25.5000, 13.0000, 24.0000, 16.0000,
        26.0000, 19.4000, 17.0000, 30.90

## Training A Deep neural network model

In [33]:
from torch.utils.data import DataLoader, TensorDataset


train_ds = TensorDataset(x_train, y_train)
batch_size = 8
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

### Creating  simple model for regression task

In [34]:
hidden_units = [8, 4]
input_size = x_train.shape[1]

all_layers = []
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 1))

model = nn.Sequential(*all_layers) # unpacking the layers



In [36]:
model

Sequential(
  (0): Linear(in_features=9, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

### Definig LOSS function - MSE and Optimizer - SGD for this problem 

In [37]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001) # Giving lr as 0.001 for better convering

### Train the model

In [38]:
torch.manual_seed(1)
num_epochs = 200
log_epochs = 20 
for epoch in range(num_epochs):
    loss_hist_train = 0
    for x_batch, y_batch in train_dl:
        pred = model(x_batch)[:, 0]
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_hist_train += loss.item()
    if epoch % log_epochs==0:
        print(f'Epoch {epoch}  Loss {loss_hist_train/len(train_dl):.4f}')

Epoch 0  Loss 536.1047
Epoch 20  Loss 8.4361
Epoch 40  Loss 7.8695
Epoch 60  Loss 7.1891
Epoch 80  Loss 6.7062
Epoch 100  Loss 6.7599
Epoch 120  Loss 6.3124
Epoch 140  Loss 6.6864
Epoch 160  Loss 6.7648
Epoch 180  Loss 6.2156


## Evaluating the model

In [39]:
with torch.no_grad(): # Do not track gradient
    pred = model(x_test.float())[:, 0]
    loss = loss_fn(pred, y_test)
    print(f'Test MSE: {loss.item():.4f}')
    print(f'Test MAE: {nn.L1Loss()(pred, y_test).item():.4f}')

Test MSE: 9.6130
Test MAE: 2.1211
